# Giải bài tập MCQ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load dữ liệu

In [ ]:
data_dir = '../Data/'
orders = pd.read_csv(data_dir + 'orders.csv', parse_dates=['order_date'], low_memory=False)
products = pd.read_csv(data_dir + 'products.csv', low_memory=False)
returns = pd.read_csv(data_dir + 'returns.csv', low_memory=False)
web_traffic = pd.read_csv(data_dir + 'web_traffic.csv', low_memory=False)
order_items = pd.read_csv(data_dir + 'order_items.csv', low_memory=False)
customers = pd.read_csv(data_dir + 'customers.csv', low_memory=False)
payments = pd.read_csv(data_dir + 'payments.csv', low_memory=False)
geography = pd.read_csv(data_dir + 'geography.csv', low_memory=False)
print('Đã tải xong toàn bộ dữ liệu.')

### Q1: Trung vị số ngày giữa hai lần mua liên tiếp

Yêu cầu: Tính trung vị khoảng cách số ngày giữa hai lần mua hàng liên tiếp của các khách hàng có nhiều hơn 1 đơn.

In [ ]:
# Sắp xếp theo khách hàng và ngày
orders_sorted = orders.sort_values(by=['customer_id', 'order_date'])

# Lọc khách hàng có nhiều hơn 1 đơn (Dùng duplicated để tối ưu hơn isin)
df_multi = orders_sorted[orders_sorted.duplicated(subset='customer_id', keep=False)].copy()

# Tính chênh lệch ngày giữa các đơn liên tiếp
df_multi['diff_days'] = df_multi.groupby('customer_id')['order_date'].diff().dt.days

# Tính trung vị
median_gap = df_multi['diff_days'].median()
print(f"Q1 Answer: {median_gap} ngày")

### Q2: Tỷ suất lợi nhuận gộp trung bình theo segment

Yêu cầu: Tìm phân khúc (segment) sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất.

In [ ]:
# Tính lợi nhuận gộp: (Price - COGS) / Price
products['margin'] = (products['price'] - products['cogs']) / products['price']

# Tính trung bình theo segment
margin_by_segment = products.groupby('segment')['margin'].mean().sort_values(ascending=False)
print("Q2 Answer:")
print(margin_by_segment.head())

### Q3: Lý do trả hàng nhiều nhất cho danh mục Streetwear

Yêu cầu: Xác định nguyên nhân trả hàng phổ biến nhất cho danh mục Streetwear.

In [ ]:
# Merge returns với products để lấy category
ret_prod = returns.merge(products, on='product_id')

# Lọc danh mục Streetwear
streetwear_returns = ret_prod[ret_prod['category'] == 'Streetwear']

# Đếm tần suất các lý do trả hàng
top_reason = streetwear_returns['return_reason'].value_counts()
print("Q3 Answer:")
print(top_reason.head())

### Q4: Nguồn traffic có tỷ lệ thoát trung bình thấp nhất

Yêu cầu: Nguồn lưu lượng (traffic_source) nào mang lại lượng khách có tỷ lệ thoát (bounce_rate) trung bình thấp nhất?

In [ ]:
# Nhóm theo nguồn traffic và tính trung bình bounce_rate
bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print("Q4 Answer:")
print(bounce_rate.head())

### Q5: Tỷ lệ order_items có khuyến mãi

Yêu cầu: Tính tỷ lệ phần trăm số dòng đơn hàng (order_items) có áp dụng mã khuyến mãi.

In [ ]:
# Đếm số lượng items có áp dụng promo_id hoặc promo_id_2
has_promo = (order_items['promo_id'].notnull() | order_items['promo_id_2'].notnull()).sum()
total_items = len(order_items)

# Tính tỷ lệ %
promo_ratio = has_promo / total_items * 100
print(f"Q5 Answer: {promo_ratio:.2f}%")

### Q6: Nhóm tuổi có số đơn hàng trung bình cao nhất

Yêu cầu: Nhóm tuổi (age_group) nào có số đơn hàng trung bình mỗi khách hàng cao nhất?

In [ ]:
# Đếm số đơn hàng của mỗi khách hàng
orders_per_cust = orders['customer_id'].value_counts().reset_index()
orders_per_cust.columns = ['customer_id', 'order_count']

# Merge với bảng khách hàng để lấy age_group
cust_orders = customers.merge(orders_per_cust, on='customer_id', how='left')
cust_orders['order_count'] = cust_orders['order_count'].fillna(0)

# Tính trung bình số đơn theo age_group
age_group_orders = cust_orders[cust_orders['age_group'].notnull()].groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print("Q6 Answer:")
print(age_group_orders.head())

### Q7: Vùng có doanh thu cao nhất

Yêu cầu: Khu vực địa lý (region) nào tạo ra nhiều doanh thu nhất?

In [ ]:
# Merge orders với customers (bỏ cột zip của orders để tránh trùng lặp khi merge)
orders_cust = orders.drop(columns=['zip']).merge(customers, on='customer_id', how='inner')
# Merge với geography để lấy region
orders_geo = orders_cust.merge(geography, on=['zip', 'city'], how='inner')
# Merge với payments để lấy doanh thu (payment_value)
orders_full = orders_geo.merge(payments, on='order_id', how='inner')

# Tính tổng doanh thu theo region
region_revenue = orders_full.groupby('region')['payment_value'].sum().sort_values(ascending=False)
print("Q7 Answer:")
print(region_revenue.head())

### Q8: Phương thức thanh toán nhiều nhất trong đơn huỷ

Yêu cầu: Phương thức thanh toán nào phổ biến nhất trong các đơn hàng bị huỷ (cancelled)?

In [ ]:
# Lọc các đơn bị hủy
cancelled_orders = orders[orders['order_status'] == 'cancelled']

# Merge với bảng payments để lấy phương thức thanh toán
cancelled_payments = cancelled_orders.merge(payments, on='order_id')

# Đếm tần suất phương thức thanh toán
top_payment = cancelled_payments['payment_method'].value_counts()
print("Q8 Answer:")
print(top_payment.head())

### Q9: Kích thước sản phẩm có tỷ lệ trả hàng cao nhất

Yêu cầu: Kích thước (size) nào có tỷ lệ trả hàng (số dòng bị trả / số dòng đã bán) cao nhất?

In [ ]:
# Đếm số lượng đã bán theo từng size
items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
size_counts = items_with_size['size'].value_counts()

# Đếm số lượng bị trả lại theo từng size
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
return_counts = returns_with_size['size'].value_counts()

# Tính tỷ lệ trả hàng
return_ratio = (return_counts / size_counts).dropna().sort_values(ascending=False)
print("Q9 Answer:")
print(return_ratio.head())

### Q10: Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

Yêu cầu: Kế hoạch trả góp (installments) nào có giá trị thanh toán trung bình (payment_value) cao nhất?

In [ ]:
# Tính trung bình giá trị thanh toán theo số tháng trả góp
avg_payment_installments = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print("Q10 Answer:")
print(avg_payment_installments.head())